# Clustering

We will group tasks using description vectors, context vectors, aggregate fields, and a combination of text and aggregates. We compare family counts, consistency across seeds, and representative tasks.

No FMUCD families have been fitted yet. Set `RUN_CLUSTERING=True` after reviewing an embedding run. Duration is not used to fit or choose the clusters.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
helper_root = Path.cwd() if (Path.cwd() / 'pipeline_utils.py').exists() else Path.cwd() / 'fmucd_analysis'
if str(helper_root) not in sys.path:
    sys.path.insert(0, str(helper_root))
from pipeline_utils import (project_root, read_json, write_json, fingerprint,
                            assert_fingerprints, require_new_run, safe_run_name)
ROOT = project_root()
INPUT_DIR = ROOT / 'data/clustering'
source_manifest = read_json(INPUT_DIR / 'manifest.json')
SEED = 42
pd.set_option('display.max_colwidth', 100)

In [2]:
RUN_CLUSTERING = False
EMBEDDING_RUN = 'tfidf_review'
RUN_NAME = 'families_review'
OVERWRITE = False
K_VALUES = [8, 16, 32]
FIT_TASK_LIMIT = 50_000
SILHOUETTE_TASK_LIMIT = 2_000
TEXT_WEIGHT = 0.5
PREDICT_BATCH = 10_000
EMBED_DIR = ROOT / 'data/embeddings' / safe_run_name(EMBEDDING_RUN)
OUT_DIR = ROOT / 'data/families' / safe_run_name(RUN_NAME)
METHODS = ['text_wo', 'text_context', 'aggregate_only', 'combined_context']

## 1. Prepare the clustering inputs

We align tasks by ID, fill missing numeric values, and scale the training profiles. Text and numeric blocks are normalized before combining them. New tasks without training history remain unassigned for aggregate-based methods (`-1`).

In [3]:
if RUN_CLUSTERING:
    import joblib
    import sklearn
    from sklearn.cluster import MiniBatchKMeans
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler, Normalizer
    from sklearn.metrics import silhouette_score, adjusted_rand_score
    if not 0 < TEXT_WEIGHT < 1:
        raise ValueError('TEXT_WEIGHT must be between zero and one.')
    emb_manifest = read_json(EMBED_DIR / 'manifest.json')
    assert_fingerprints(INPUT_DIR, emb_manifest['source_fingerprints'])
    assert_fingerprints(EMBED_DIR, emb_manifest['artifact_fingerprints'])
    require_new_run(OUT_DIR, OVERWRITE)
    catalog = pd.read_parquet(EMBED_DIR / 'catalog.parquet')
    assert catalog['task_id'].is_unique
    aggregate_columns = source_manifest['aggregate_feature_columns']
    aggregate_table = pd.read_parquet(INPUT_DIR / 'tasks_aggregates_train.parquet')
    aligned = catalog[['task_id']].merge(aggregate_table, on='task_id', how='left', validate='one_to_one', indicator=True)

In [4]:
if RUN_CLUSTERING:
    has_aggregate = aligned['_merge'].eq('both').to_numpy()
    is_train = catalog['is_training_task'].to_numpy(dtype=bool)
    assert has_aggregate[is_train].all()
    aggregate_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median', keep_empty_features=True, add_indicator=True)),
        ('scale', StandardScaler()), ('normalize', Normalizer())])
    aggregate_transformer.fit(aligned.loc[is_train, aggregate_columns])
    numeric = aggregate_transformer.transform(aligned[aggregate_columns]).astype('float32')
    joblib.dump(aggregate_transformer, OUT_DIR / 'aggregate_transformer.joblib')
    text = {v: np.load(EMBED_DIR / emb_manifest['variants'][v]['file'], mmap_mode='r')
            for v in ['text_wo', 'text_context']}
    assert all(len(x) == len(catalog) for x in text.values())

In [5]:
if RUN_CLUSTERING:
    def block(method, indices):
        if method in text:
            return np.asarray(text[method][indices], dtype=np.float32)
        if method == 'aggregate_only':
            return numeric[indices]
        return np.hstack([np.sqrt(TEXT_WEIGHT)*text['text_context'][indices],
                          np.sqrt(1-TEXT_WEIGHT)*numeric[indices]]).astype('float32')

In [6]:
if RUN_CLUSTERING:
    def eligible_indices(method):
        mask = np.ones(len(catalog), dtype=bool) if method in text else has_aggregate.copy()
        if method in text:
            norms = np.linalg.norm(text[method], axis=1)
            mask &= norms > 1e-8
        return np.flatnonzero(mask)

## 2. Compare family counts

We compare K-means solutions using silhouette, cluster sizes, and agreement across two seeds. We will inspect the work descriptions before deciding whether the families are useful.

In [7]:
if RUN_CLUSTERING:
    rng = np.random.default_rng(SEED)
    train_indices = np.flatnonzero(is_train)
    common_fit = np.sort(rng.choice(train_indices, min(FIT_TASK_LIMIT, len(train_indices)), replace=False))
    candidates, fitted_candidates = [], {}

In [8]:
if RUN_CLUSTERING:
    for method in METHODS:
        allowed = eligible_indices(method)
        fit_indices = np.intersect1d(common_fit, allowed)
        X_fit = block(method, fit_indices)
        evaluation = np.sort(rng.choice(len(X_fit), min(SILHOUETTE_TASK_LIMIT, len(X_fit)), replace=False))
        for k in K_VALUES:
            if not 1 < k < len(X_fit):
                continue
            first = MiniBatchKMeans(n_clusters=k, random_state=SEED, n_init=3, batch_size=2048)
            second = MiniBatchKMeans(n_clusters=k, random_state=SEED+1, n_init=3, batch_size=2048)
            first.fit(X_fit); second.fit(X_fit)
            labels = first.predict(X_fit)
            counts = np.bincount(labels, minlength=k)
            sample_labels = labels[evaluation]
            score = (silhouette_score(X_fit[evaluation], sample_labels)
                     if 1 < len(np.unique(sample_labels)) < len(evaluation) else np.nan)
            candidates.append({'method': method, 'k': k, 'silhouette': score,
                'seed_ari': adjusted_rand_score(sample_labels, second.predict(X_fit[evaluation])),
                'largest_cluster_share': float(counts.max()/counts.sum()),
                'empty_clusters': int((counts == 0).sum()), 'minimum_cluster_tasks': int(counts.min()),
                'fit_tasks': len(X_fit)})
            fitted_candidates[(method, k)] = first

In [9]:
if RUN_CLUSTERING:
    candidate_scores = pd.DataFrame(candidates)
    if candidate_scores.empty:
        raise ValueError('No valid cluster-count candidates for this training sample.')
    display(candidate_scores.round(3))

In [10]:
if RUN_CLUSTERING:
    candidate_scores.to_csv(OUT_DIR / 'candidate_scores.csv', index=False)

In [11]:
if RUN_CLUSTERING:
    for method, group in candidate_scores.groupby('method'):
        plt.plot(group['k'], group['silhouette'], marker='o', label=method)
    plt.xlabel('Family count')
    plt.ylabel('Sample silhouette')
    plt.title('Training-only cluster comparison')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 3. Inspect and save the families

The initial choice is the highest-silhouette candidate with no empty clusters. `SELECTED_K` lets us change that choice after checking representative tasks. Family numbers are labels, not ordered values.

In [12]:
SELECTED_K = {}

In [13]:
if RUN_CLUSTERING:
    assignments = catalog[['task_id', 'is_training_task']].copy()
    representatives, selected = [], {}

In [14]:
if RUN_CLUSTERING:
    for method, group in candidate_scores.groupby('method'):
        choices = group.loc[group['empty_clusters'].eq(0) & group['silhouette'].notna()]
        if choices.empty:
            raise ValueError(f'No nonempty, scored candidate for {method}. Review K_VALUES.')
        k = int(SELECTED_K.get(method, choices.sort_values(['silhouette', 'seed_ari'], ascending=False).iloc[0]['k']))
        if (method, k) not in fitted_candidates:
            raise ValueError(f'{method}: selected k was not fitted.')
        model = fitted_candidates[(method, k)]
        allowed = eligible_indices(method)
        labels = np.full(len(catalog), -1, dtype=np.int32)
        for start in range(0, len(allowed), PREDICT_BATCH):
            indices = allowed[start:start+PREDICT_BATCH]
            labels[indices] = model.predict(block(method, indices))
        assignments[f'family_{method}'] = labels
        joblib.dump(model, OUT_DIR / f'{method}_kmeans.joblib')
        selected[method] = {'k': k, 'assigned_tasks': int((labels >= 0).sum())}
        # Bound the review sample without computing a full all-task distance matrix.
        reference = np.intersect1d(common_fit, allowed)
        reference_vectors = block(method, reference)
        distances = np.linalg.norm(reference_vectors - model.cluster_centers_[labels[reference]], axis=1)
        examples = catalog.iloc[reference][['task_id', 'text_wo']].copy()
        examples['family'] = labels[reference]; examples['centroid_distance'] = distances
        examples['method'] = method
        representatives.append(examples.sort_values('centroid_distance').groupby('family', sort=False).head(3))

In [15]:
if RUN_CLUSTERING:
    representatives = pd.concat(representatives, ignore_index=True)
    assignments.to_parquet(OUT_DIR / 'task_families.parquet', index=False)
    representatives.to_parquet(OUT_DIR / 'representatives.parquet', index=False)
    display(pd.DataFrame(selected).T)

In [16]:
if RUN_CLUSTERING:
    display(representatives.groupby('method', sort=False).head(9))

In [17]:
if RUN_CLUSTERING:
    family_manifest = {'stage': 'families', 'run_name': RUN_NAME, 'embedding_run': EMBEDDING_RUN,
        'selected': selected, 'text_weight': TEXT_WEIGHT, 'fit_task_limit': FIT_TASK_LIMIT,
        'seed': SEED, 'family_columns': [f'family_{m}' for m in selected],
        'source_fingerprints': fingerprint([INPUT_DIR / 'manifest.json', INPUT_DIR / 'tasks_aggregates_train.parquet']),
        'embedding_manifest_fingerprint': fingerprint([EMBED_DIR / 'manifest.json']),
        'artifact_fingerprints': fingerprint([OUT_DIR / 'task_families.parquet']),
        'unassigned_label': -1, 'selection': 'training silhouette, nonempty clusters, manual overrides if provided',
        'versions': {'sklearn': sklearn.__version__}}
    write_json(OUT_DIR / 'manifest.json', family_manifest)

## What we will take forward

We will use one comparison plot and a few representative families in the presentation. We also need to report how many tasks could be assigned, especially when a method requires prior history. All numeric profiles must be rebuilt if the training window changes.